# 第5章：注意力机制的直觉理解

> **预计学习时间：约 30 分钟**
>
> 注意力机制是 Transformer 的核心，也是理解所有现代大模型的关键。本章用搜索引擎和数据库查询的类比，帮你建立对注意力机制的直觉理解，然后亲手从零实现 Scaled Dot-Product Attention 并可视化注意力权重。

**运行环境**: Kaggle Notebook (CPU 即可)
**前置要求**: 完成第4章词嵌入基础

---
## 0. 环境准备

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F  # 包含 softmax、relu 等常用函数
import numpy as np
import matplotlib.pyplot as plt
import math  # 用于 sqrt 计算缩放因子

# 设置随机种子，确保每次运行结果一致
torch.manual_seed(42)
np.random.seed(42)

plt.rcParams['font.size'] = 12

print(f"PyTorch 版本: {torch.__version__}")
print("本章主要使用 CPU，无需 GPU")

---
## 1. 为什么需要注意力机制？

### 没有注意力的世界：信息瓶颈

在第1章中我们学过，RNN/LSTM 逐字处理文本，把整个句子压缩成**一个固定大小的向量**。这就像：

> 你读了一本 500 页的书，但只被允许用**一张便利贴**来概括全部内容，然后回答关于这本书的问题。
>
> 无论你多聪明，一张便利贴都装不下 500 页的信息！

```
没有注意力的 RNN：

  "The cat sat on the mat because it was tired"
       |     |    |   |   |    |       |   |    |
       v     v    v   v   v    v       v   v    v
      [h1]→[h2]→[h3]→[h4]→[h5]→[h6]→[h7]→[h8]→[h9]
                                                  |
                                                  v
                                          [单一向量]  ← 所有信息都被压缩到这里！
                                                  |
                                                  v
                                              [解码器]

  问题："it" 指代的是 "cat" —— 但当我们读到 "it" 时，
  "cat" 的信息在这个单一向量中已经逐渐消失了！
```

### 注意力的解决方案：随时回头看

注意力机制允许模型在处理每个词时，**直接回头查看所有之前的词**，而不是只依赖一个压缩的向量。

```
有了注意力机制：

  "The cat sat on the mat because it was tired"

  处理 "it" 时，模型可以：
    - 回头看 "cat"  → 高注意力 (0.65)
    - 回头看 "mat"  → 中等注意力 (0.20)
    - 回头看 "The"  → 低注意力 (0.05)
    - ...

  模型现在知道 "it" 指代的是 "cat"！
```

### 类比：搜索引擎

![注意力机制类比](images/attention_analogy.png)

*图1：注意力机制的搜索引擎类比*

注意力机制的工作方式非常像搜索引擎：

| 搜索引擎 | 注意力机制 |
|----------|-----------|
| 你输入一个**搜索词** | 当前词生成一个 **Query（查询）** |
| 搜索引擎有很多**网页标题** | 所有词生成 **Key（关键词）** |
| 搜索引擎计算**相关度排名** | 计算 Query 和每个 Key 的**相似度** |
| 你看到排名最高的**网页内容** | 根据相似度加权获取 **Value（内容）** |

这就是注意力机制的核心思想：**用 Query 去搜索最相关的 Key，然后取出对应的 Value**。

> 💡 **记忆要点**
> - 没有注意力：所有信息被压缩成一个向量 → 信息瓶颈
> - 有注意力：每个词都能**直接回头查看**所有其他词
> - 注意力 ≈ 搜索引擎：Query（搜索词）→ Key（网页标题）→ Value（网页内容）

---
## 2. Q、K、V：注意力的三要素

### 类比：数据库查询

Q、K、V 的概念直接对应数据库查询操作：

```
数据库查询类比：

  SELECT value FROM table WHERE key MATCHES query

  Query (Q) = 你在找什么？
  Key   (K) = 每条记录的标签是什么？
  Value (V) = 每条记录包含什么内容？

举例：
  你想理解 "The cat sat ... because it was tired" 中的 "it"

  Query:  "it" 问 → "我指代的是谁？"
  Keys:   "The"→冠词, "cat"→动物, "sat"→动作, "mat"→物体, ...
  Values: 每个词的实际内容/含义

  匹配: Q("it") 和 K("cat") 最匹配
  结果: 以高权重返回 V("cat") → "it" 学到了自己指代 "cat"
```

### 实际的计算过程

在神经网络中，Q、K、V 都是通过**线性变换**从输入向量计算出来的：

```
输入词向量: x  (形状: d_model)

  Q = x @ W_Q    ← "我在找什么？"
  K = x @ W_K    ← "我包含什么？"
  V = x @ W_V    ← "我携带什么信息？"

  W_Q、W_K、W_V 是可学习的权重矩阵！
  模型会学会该查询什么、该匹配什么、该提取什么。
```

### 注意力分数的计算步骤

| 步骤 | 操作 | 类比 |
|------|------|------|
| 1 | 计算 Q 和 K 的点积 | 搜索词和网页标题的相关度 |
| 2 | 除以 sqrt(d_k) 缩放 | 防止分数太大导致 softmax 饱和 |
| 3 | Softmax 归一化 | 把分数转换成概率分布（总和=1） |
| 4 | 用注意力权重加权 V | 按相关度提取信息 |

公式：

> **Attention(Q, K, V) = softmax(Q @ K^T / sqrt(d_k)) @ V**

这就是 Scaled Dot-Product Attention 的全部内容！

> 💡 **记忆要点**
> - **Q (Query)** = 我在找什么？**K (Key)** = 我有什么标签？**V (Value)** = 我携带什么信息？
> - Q、K、V 都是通过**可学习的线性变换**从输入向量计算出来的
> - 注意力公式：**softmax(QK^T / sqrt(d_k)) @ V**
> - 这个公式做的事：用 Q 搜索最相关的 K，然后按相关度提取 V

---
## 3. 从零实现 Scaled Dot-Product Attention

让我们一步一步手动实现注意力机制，理解每个步骤在做什么。

In [ ]:
# =============================================
# 3.1 准备输入数据
# =============================================

# 假设我们有一个 4 个词的句子，每个词用 8 维向量表示
seq_len = 4
d_model = 8

# 模拟输入（4 个词的嵌入向量）
# 假设句子是 "The cat sat there"
torch.manual_seed(42)
X = torch.randn(seq_len, d_model)

print(f"输入 X 的形状: {X.shape}  [seq_len={seq_len}, d_model={d_model}]")
print(f"含义: {seq_len} 个词，每个词用 {d_model} 维向量表示")
print(f"\n输入矩阵 X:")
print(X)

In [ ]:
# =============================================
# 3.2 生成 Q, K, V
# =============================================

d_k = d_model  # Q, K 的维度（通常等于 d_model 或 d_model/n_heads）
d_v = d_model  # V 的维度

# 创建可学习的权重矩阵
W_Q = nn.Linear(d_model, d_k, bias=False)
W_K = nn.Linear(d_model, d_k, bias=False)
W_V = nn.Linear(d_model, d_v, bias=False)

# 计算 Q, K, V
Q = W_Q(X)  # [seq_len, d_k]
K = W_K(X)  # [seq_len, d_k]
V = W_V(X)  # [seq_len, d_v]

print("线性变换: X → Q, K, V")
print(f"  W_Q: ({d_model}, {d_k})  → Q: {Q.shape}")
print(f"  W_K: ({d_model}, {d_k})  → K: {K.shape}")
print(f"  W_V: ({d_model}, {d_v})  → V: {V.shape}")
print(f"\n每个词都有自己的 Q、K、V 向量")
print(f"  Q = '我在找什么'")
print(f"  K = '我有什么标签'")
print(f"  V = '我携带什么信息'")

### 为什么需要点积
点积结果越大 → 两个向量越相似（方向越接近）
点积结果越小 → 两个向量越不相似

[1, 0]·[1, 0] = 1  （完全相同，最大）
[1, 0]·[0, 1] = 0  （完全不同，正交）
[1, 0]·[-1,0] = -1 （完全相反，最小）

### Q @ K.T 的计算含义
Q @ K.T = 每个Query 和 每个Key 做点积
         K.T
         词1  词2  词3  词4
    词1 [  ·    ·    ·    · ]
Q   词2 [  ·    ·    ·    · ]
    词3 [  ·    ·    ·    · ]
    词4 [  ·    ·    ·    · ]

scores[i][j] = Q[i] 和 K[j] 的点积
             = 第i个词 对 第j个词 的关注程度

In [ ]:
# =============================================
# 3.3 Step 1: 计算注意力分数 (Q @ K^T)
# =============================================

# Q @ K^T: 每个 Query 和每个 Key 的点积
scores = Q @ K.T  # [seq_len, seq_len]

print("Step 1: 注意力分数 = Q @ K^T")
print(f"  Q: {Q.shape} @ K^T: {K.T.shape} → scores: {scores.shape}")
print(f"\n分数矩阵 (4x4):")
print(f"  scores[i][j] = 第 i 个词的 Query 和第 j 个词的 Key 的相关度")
print(f"\n{scores.data}")
print(f"\n比如 scores[0][1] = {scores[0][1].item():.4f}")
print(f"  表示第 0 个词（The）的 Query 和第 1 个词（cat）的 Key 的相关度")

In [ ]:
# =============================================
# 3.4 Step 2: 缩放 (除以 sqrt(d_k))
# =============================================

# 为什么要缩放？
# 当 d_k 很大时，点积值会变得很大
# 大的值经过 softmax 后会变成接近 0 和 1 的极端值
# 梯度会变得很小，难以训练

scale = math.sqrt(d_k)
scaled_scores = scores / scale

print(f"Step 2: 缩放 — 除以 sqrt(d_k) = sqrt({d_k}) = {scale:.2f}")
print(f"\n缩放前分数范围: [{scores.min().item():.4f}, {scores.max().item():.4f}]")
print(f"缩放后分数范围: [{scaled_scores.min().item():.4f}, {scaled_scores.max().item():.4f}]")
print(f"\n缩放后的分数:")
print(scaled_scores.data)

# 演示为什么需要缩放
print(f"\n--- 为什么需要缩放？看 softmax 的行为 ---")
big_scores = torch.tensor([10.0, 20.0, 30.0])
small_scores = torch.tensor([1.0, 2.0, 3.0])
print(f"  大分数 softmax({big_scores.tolist()}):   {F.softmax(big_scores, dim=0).tolist()}")
print(f"  小分数 softmax({small_scores.tolist()}): {F.softmax(small_scores, dim=0).tolist()}")
print(f"\n  大分数时 softmax 几乎是 one-hot（全部注意力集中在一个词上）")
print(f"  小分数时 softmax 更平滑（注意力分散在多个词上）")
print(f"  缩放让分数保持在合理范围，避免 softmax 过于极端")

In [ ]:
# =============================================
# 3.5 Step 3: Softmax 归一化
# =============================================

attention_weights = F.softmax(scaled_scores, dim=-1)  # 沿最后一维归一化

print("Step 3: Softmax 归一化")
print(f"  每一行的权重之和 = 1（概率分布）")
print(f"\nAttention Weight矩阵:")
print(attention_weights.data)
print(f"\n每行之和: {attention_weights.sum(dim=-1).tolist()}")
print(f"\n解读:")
print(f"  attention_weights[i][j] = 第 i 个词对第 j 个词的注意力")
print(f"  值越大 → 越关注，值越小 → 越忽略")

In [ ]:
# =============================================
# 3.6 Step 4: 加权求和 (attention_weights @ V)
# =============================================

output = attention_weights @ V  # [seq_len, d_v]

print("Step 4: 加权求和 — 按Attention Weight提取 Value")
print(f"  attention_weights: {attention_weights.shape} @ V: {V.shape} → output: {output.shape}")
print(f"\n输出的每一行 = 所有 Value 向量的加权和")
print(f"  output[i] = sum(attention_weights[i][j] * V[j] for all j)")
print(f"\n注意力关注谁多，就从谁那里获取更多信息!")
print(f"\n最终输出:")
print(output.data)

In [ ]:
# =============================================
# 3.7 封装成完整函数
# =============================================

def scaled_dot_product_attention(Q, K, V, mask=None):
    """Scaled Dot-Product Attention

    Args:
        Q: Query [seq_len, d_k]
        K: Key   [seq_len, d_k]
        V: Value [seq_len, d_v]
        mask: optional mask [seq_len, seq_len]

    Returns:
        output: [seq_len, d_v]
        attention_weights: [seq_len, seq_len]
    """
    d_k = Q.shape[-1]

    # Step 1: 计算分数
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)

    # Optional: 应用 mask（后面会讲）
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    # Step 2: Softmax
    attention_weights = F.softmax(scores, dim=-1)

    # Step 3: 加权求和
    output = attention_weights @ V

    return output, attention_weights


# 测试
output, weights = scaled_dot_product_attention(Q, K, V)
print("封装后的 Attention 函数:")
print(f"  输入: Q{Q.shape}, K{K.shape}, V{V.shape}")
print(f"  输出: output{output.shape}, weights{weights.shape}")
print(f"\n完整公式: Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) @ V")
print(f"就这三步! 整个注意力机制的计算核心就在这里。")

---
## 4. 可视化注意力权重

注意力权重矩阵告诉我们模型在处理每个词时"关注"了哪些其他词。热力图是最直观的可视化方式。

In [ ]:
# =============================================
# 4.1 可视化注意力热力图
# =============================================
# Attention Weight矩阵 weights[i][j] = 第 i 个词对第 j 个词的关注程度
# 用热力图（Heatmap）来展示：颜色越深 = 注意力越高

words = ["The", "cat", "sat", "there"]

fig, ax = plt.subplots(figsize=(8, 6))

# imshow 把矩阵画成彩色网格图
# cmap='Blues' 表示用蓝色系，数值越大颜色越深
im = ax.imshow(weights.detach().numpy(), cmap='Blues', aspect='auto')

# 设置坐标轴标签（用词代替数字索引）
ax.set_xticks(range(len(words)))
ax.set_yticks(range(len(words)))
ax.set_xticklabels(words, fontsize=13)
ax.set_yticklabels(words, fontsize=13)
ax.set_xlabel('Key (attended to)', fontsize=12)
ax.set_ylabel('Query (attending)', fontsize=12)

# 在每个格子里标注具体数值
for i in range(len(words)):
    for j in range(len(words)):
        val = weights[i, j].item()
        # 深色背景用白色字，浅色背景用黑色字，保证可读性
        color = 'white' if val > 0.4 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                fontsize=12, color=color, fontweight='bold')

ax.set_title('Attention Weight Heatmap', fontsize=14)
plt.colorbar(im, ax=ax, label='Attention Weight')
plt.tight_layout()
plt.show()

print("如何阅读这张热力图:")
print("  - 行 = Query（正在处理的词）")
print("  - 列 = Key（被关注的词）")
print("  - 颜色越深 = 注意力越高 = 越关注那个词")
print("  - 每行之和 = 1.0（概率分布）")

In [ ]:
# =============================================
# 4.2 用一个更有意义的例子
# =============================================

# 手工构造一个注意力矩阵，展示直觉
# 句子: "The animal didn't cross the street because it was tired"
demo_words = ["The", "animal", "didn't", "cross", "the", "street", "because", "it", "was", "tired"]
n = len(demo_words)

# 构造一个直觉上合理的Attention Weight（针对 "it" 这个词）
# "it" 应该最关注 "animal"
it_attention = torch.tensor([0.03, 0.45, 0.02, 0.05, 0.03, 0.12, 0.05, 0.05, 0.05, 0.15])
it_attention = it_attention / it_attention.sum()  # 归一化

fig, ax = plt.subplots(figsize=(14, 3))

colors = plt.cm.Reds(it_attention.numpy())
bars = ax.bar(range(n), it_attention.numpy(), color=colors, edgecolor='gray', linewidth=0.5)

ax.set_xticks(range(n))
ax.set_xticklabels(demo_words, fontsize=12, rotation=0)
ax.set_ylabel('Attention Weight', fontsize=12)
ax.set_title('"it" Attention Distribution — What does "it" refer to?', fontsize=14)

# 标注最高的
max_idx = it_attention.argmax()
ax.annotate(f'{it_attention[max_idx]:.2f}',
            xy=(max_idx, it_attention[max_idx]),
            xytext=(max_idx, it_attention[max_idx] + 0.06),
            fontsize=13, fontweight='bold', ha='center', color='red',
            arrowprops=dict(arrowstyle='->', color='red'))

plt.tight_layout()
plt.show()

print('"it" 最关注 "animal"（注意力 = 0.45）')
print('这就是注意力机制的直觉：模型学会了 "it" 指代的是 "animal"')
print('这种指代关系（coreference）是 RNN 很难处理的，但注意力机制可以轻松捕获')

---
## 5. Self-Attention：句子和自己做注意力

### 什么是 Self-Attention？

在 Self-Attention 中，Q、K、V **全部来自同一个输入序列**。每个词都既是提问者（Query），也是被查询者（Key），也是信息提供者（Value）。

```
普通注意力（Cross-Attention）:
  Q 来自：解码器
  K, V 来自：编码器
  → 解码器去查询编码器的信息

自注意力（Self-Attention）:
  Q, K, V 全部来自：同一个序列
  → 每个词都会关注同一句话中的所有其他词

例子："I love deep learning"

  "I"        问："谁和我有关？"
  "love"     问："谁和我有关？"
  "deep"     问："谁和我有关？"
  "learning" 问："谁和我有关？"

  每个词都会查看所有词（包括自己）
  来收集相关的上下文信息。
```

### 为什么 Self-Attention 这么强大？

| 特性 | RNN | Self-Attention |
|------|-----|---------------|
| 远距离依赖 | 需要逐步传递，容易遗忘 | 直接连接任意两个词 |
| 并行计算 | 必须串行（等前一步完成） | 完全并行 |
| 最长路径 | O(n)，序列越长越难 | O(1)，任意两词直接连接 |

**动手验证**：让我们实现一个完整的 Self-Attention 模块。

In [ ]:
# =============================================
# 5.1 实现 Self-Attention 模块
# =============================================

class SelfAttention(nn.Module):
    """Self-Attention 模块

    Q, K, V 都来自同一个输入序列（所以叫"自"注意力）
    和 Cross-Attention 的区别：Cross-Attention 的 Q 来自解码器，K/V 来自编码器

    内部结构：
      W_Q: 把输入变成 Query（"我在找什么"）
      W_K: 把输入变成 Key（"我的标签是什么"）
      W_V: 把输入变成 Value（"我的内容是什么"）
    三个权重矩阵都是可学习的，通过训练自动调整
    """
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        # 三个线性层，分别把输入投影到 Q、K、V 空间
        # bias=False 是因为 Transformer 原始论文中不使用偏置
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        """
        x: [batch_size, seq_len, d_model] — 输入序列
        mask: [batch_size, seq_len, seq_len] — 可选的注意力遮罩
        返回: output（同形状的输出）, weights（Attention Weight矩阵）
        """
        # 用三个不同的线性变换，从同一个输入 x 生成 Q、K、V
        Q = self.W_Q(x)  # [batch, seq_len, d_model]
        K = self.W_K(x)  # [batch, seq_len, d_model]
        V = self.W_V(x)  # [batch, seq_len, d_model]

        # 调用之前实现的 Scaled Dot-Product Attention
        output, weights = scaled_dot_product_attention(Q, K, V, mask)
        return output, weights


# 创建 Self-Attention 模块
d_model = 16  # 嵌入维度（实际中 GPT-2 用 768，这里用小值便于观察）
self_attn = SelfAttention(d_model)

# 模拟一个 batch 的输入
batch_size = 1
seq_len = 6   # 句子长度 6 个词
x = torch.randn(batch_size, seq_len, d_model)

# 前向传播
output, attn_weights = self_attn(x)

print(f"Self-Attention 模块:")
print(f"  输入:       {x.shape}  [batch={batch_size}, seq_len={seq_len}, d_model={d_model}]")
print(f"  输出:       {output.shape}  [同样的形状!]")
print(f"  Attention Weight: {attn_weights.shape}  [batch, seq_len, seq_len]")

# 统计参数量：三个 d_model×d_model 的矩阵
total_params = sum(p.numel() for p in self_attn.parameters())
print(f"\n参数量: {total_params}")
print(f"  W_Q: {d_model}x{d_model} = {d_model**2}")
print(f"  W_K: {d_model}x{d_model} = {d_model**2}")
print(f"  W_V: {d_model}x{d_model} = {d_model**2}")
print(f"  总计: 3 x {d_model}^2 = {3 * d_model**2}")

In [ ]:
# =============================================
# 5.2 用有意义的词嵌入测试 Self-Attention
# =============================================
# 上面用的是随机输入，现在我们手工设计一些词嵌入
# 让语义相近的词（如 "The"/"the"/"on"）向量接近
# 观察 Self-Attention 是否会更关注语义相近的词

d = 16
torch.manual_seed(42)

# 创建词向量：同类词用相近的基础向量 + 小噪声
# 这样同类词的向量会很接近
base_animal = torch.randn(d)  # 动物类的基础向量
base_action = torch.randn(d)  # 动作类的基础向量
base_other = torch.randn(d)   # 功能词的基础向量

word_embeds = {
    "The":      base_other + 0.1 * torch.randn(d),   # 功能词
    "cat":      base_animal + 0.1 * torch.randn(d),   # 动物
    "sat":      base_action + 0.1 * torch.randn(d),   # 动作
    "on":       base_other + 0.15 * torch.randn(d),   # 功能词
    "the":      base_other + 0.1 * torch.randn(d),    # 功能词
    "mat":      base_other + 0.2 * torch.randn(d),    # 物体（但归入功能词类）
}

sentence = ["The", "cat", "sat", "on", "the", "mat"]
# stack: 把多个向量堆叠成矩阵, unsqueeze(0): 添加 batch 维度
x = torch.stack([word_embeds[w] for w in sentence]).unsqueeze(0)  # [1, 6, d]

# 运行 Self-Attention
self_attn_demo = SelfAttention(d)
output, weights = self_attn_demo(x)

# 可视化Attention Weight
fig, ax = plt.subplots(figsize=(8, 6))
w = weights[0].detach().numpy()  # [0] 取第一个样本, detach() 断开梯度
im = ax.imshow(w, cmap='Blues', aspect='auto')

ax.set_xticks(range(len(sentence)))
ax.set_yticks(range(len(sentence)))
ax.set_xticklabels(sentence, fontsize=13)
ax.set_yticklabels(sentence, fontsize=13)
ax.set_xlabel('Key (attended to)', fontsize=12)
ax.set_ylabel('Query (attending)', fontsize=12)

# 标注每个格子的数值
for i in range(len(sentence)):
    for j in range(len(sentence)):
        color = 'white' if w[i, j] > 0.3 else 'black'
        ax.text(j, i, f'{w[i,j]:.2f}', ha='center', va='center',
                fontsize=11, color=color)

ax.set_title('Self-Attention：\"The cat sat on the mat\"', fontsize=14)
plt.colorbar(im, ax=ax, label='Attention Weight')
plt.tight_layout()
plt.show()

print("Self-Attention 让每个词都能看到句子中的所有其他词")
print("注意模型还没训练，所以权重是随机初始化的")
print("训练后，权重会学到有意义的注意力模式")

---
## 6. Masked Attention：让模型不能"偷看未来"

在 GPT 这样的自回归模型中，生成第 N 个词时**只能看到前 N-1 个词**，不能看到后面的词。这就需要用 **Mask（遮罩）** 来屏蔽未来的信息。

```
生成 "The cat sat on the mat"：

  预测 "sat" 时，模型只能看到：
    "The" ✓  "cat" ✓  "sat" ✓  "on" ✗  "the" ✗  "mat" ✗

  注意力遮罩（1 = 可见, 0 = 屏蔽）：

           The  cat  sat   on  the  mat
    The  [  1    0    0    0    0    0  ]
    cat  [  1    1    0    0    0    0  ]
    sat  [  1    1    1    0    0    0  ]
    on   [  1    1    1    1    0    0  ]
    the  [  1    1    1    1    1    0  ]
    mat  [  1    1    1    1    1    1  ]

  这个下三角矩阵叫做"因果遮罩"（causal mask），
  因为它阻止信息从未来流向过去。
```

In [ ]:
# =============================================
# 6.1 实现 Causal Mask
# =============================================

def create_causal_mask(seq_len):
    """创建因果遮罩（下三角矩阵）"""
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask

seq_len = 6
mask = create_causal_mask(seq_len)
print(f"因果遮罩 ({seq_len}x{seq_len}):")
print(mask)
print(f"\n1 = 可以看到, 0 = 被屏蔽")
print(f"下三角矩阵：每个词只能看到自己和之前的词")

In [ ]:
# =============================================
# 6.2 对比有无 Mask 的Attention Weight
# =============================================
# 左图（无 Mask）：每个词可以看到所有词，包括"未来"的词 → BERT 的做法
# 右图（有 Mask）：每个词只能看到自己和之前的词 → GPT 的做法

sentence = ["The", "cat", "sat", "on", "the", "mat"]
x = torch.stack([word_embeds[w] for w in sentence]).unsqueeze(0)

# 无 Mask 的注意力
_, weights_no_mask = self_attn_demo(x, mask=None)

# 有因果 Mask 的注意力
# unsqueeze(0) 添加 batch 维度，让 mask 形状匹配 [batch, seq_len, seq_len]
mask = create_causal_mask(len(sentence)).unsqueeze(0)  # [1, 6, 6]
_, weights_masked = self_attn_demo(x, mask=mask)

# 画两张对比图
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, w, title in zip(axes,
                         [weights_no_mask[0].detach().numpy(), weights_masked[0].detach().numpy()],
                         ['No Mask (Bidirectional)', 'Causal Mask (Left-to-Right)']):
    im = ax.imshow(w, cmap='Blues', aspect='auto', vmin=0, vmax=0.5)
    ax.set_xticks(range(len(sentence)))
    ax.set_yticks(range(len(sentence)))
    ax.set_xticklabels(sentence, fontsize=12)
    ax.set_yticklabels(sentence, fontsize=12)
    ax.set_xlabel('Key (attended to)', fontsize=11)
    ax.set_ylabel('Query (attending)', fontsize=11)
    ax.set_title(title, fontsize=13)
    # 标注数值
    for i in range(len(sentence)):
        for j in range(len(sentence)):
            color = 'white' if w[i, j] > 0.25 else 'black'
            ax.text(j, i, f'{w[i,j]:.2f}', ha='center', va='center', fontsize=10, color=color)

plt.tight_layout()
plt.show()

print("左图（无 Mask）: 每个词可以看到所有词（包括未来的词）→ 用于 BERT")
print("右图（有 Mask）: 每个词只能看到自己和之前的词  → 用于 GPT")
print("\n右图的右上角全是 0.00 — 未来的信息被完全屏蔽了!")

---
## 7. 真实模型中的注意力模式

训练好的模型会学到不同的注意力模式。让我们模拟几种常见的模式：

In [ ]:
# =============================================
# 7.1 常见的注意力模式
# =============================================

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
words = ["I", "love", "deep", "learning", "very", "much"]
n = len(words)

# 模式 1: 关注自己（Identity）
pattern1 = torch.eye(n)
axes[0].set_title('Self-Focus\n(attend to self)', fontsize=12)

# 模式 2: 关注前一个词
pattern2 = torch.zeros(n, n)
for i in range(n):
    if i > 0:
        pattern2[i, i-1] = 0.6
    pattern2[i, i] = 0.4
axes[1].set_title('Previous Token\n(local context)', fontsize=12)

# 模式 3: 关注第一个词（全局锚点）
pattern3 = torch.zeros(n, n)
for i in range(n):
    pattern3[i, 0] = 0.5
    pattern3[i, i] = 0.3
    remaining = 0.2 / max(n - 2, 1)
    for j in range(n):
        if j != 0 and j != i:
            pattern3[i, j] = remaining
axes[2].set_title('First Token\n(global anchor)', fontsize=12)

# 模式 4: 语法关系（手工构造）
pattern4 = torch.full((n, n), 0.05)
# "I" <-> "love" (主谓)
pattern4[0, 1] = 0.4; pattern4[1, 0] = 0.4
# "deep" <-> "learning" (修饰)
pattern4[2, 3] = 0.5; pattern4[3, 2] = 0.5
# "very" <-> "much" (程度)
pattern4[4, 5] = 0.4; pattern4[5, 4] = 0.4
# 归一化每行
pattern4 = pattern4 / pattern4.sum(dim=-1, keepdim=True)
axes[3].set_title('Syntactic\n(grammar structure)', fontsize=12)

patterns = [pattern1, pattern2, pattern3, pattern4]

for ax, pattern in zip(axes, patterns):
    im = ax.imshow(pattern.numpy(), cmap='Blues', aspect='auto', vmin=0, vmax=0.7)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(words, fontsize=10, rotation=45)
    ax.set_yticklabels(words, fontsize=10)

plt.tight_layout()
plt.show()

print("训练好的 Transformer 模型中，不同层、不同头会学到不同的注意力模式：")
print("  1. Self-Focus: 关注自身信息")
print("  2. Previous Token: 关注局部上下文")
print("  3. First Token: 用第一个位置做全局信息汇总")
print("  4. Syntactic: 关注语法相关的词（主谓、修饰、并列等）")
print("\n这些模式是模型自动学到的，不需要手工设计!")

---
## 8. 从单头到多头：为什么需要 Multi-Head Attention

### 单头的局限

一个注意力头只能学一种关注模式。但理解语言需要同时关注多种关系：

| 关系类型 | 例子 |
|----------|------|
| 语法关系 | "cat" → "sat"（主谓） |
| 指代关系 | "it" → "cat"（代词指代） |
| 修饰关系 | "black" → "cat"（形容词修饰名词） |
| 位置关系 | "on" → "mat"（介词搭配） |

**解决方案：多头注意力** — 同时运行多个注意力头，每个头关注不同的关系！

```
多头注意力：

  输入 x
    |
    +---→ 头 1：学习语法关系  (Q1, K1, V1)
    |
    +---→ 头 2：学习指代关系  (Q2, K2, V2)
    |
    +---→ 头 3：学习位置关系  (Q3, K3, V3)
    |
    +---→ 头 4：学习修饰关系  (Q4, K4, V4)
    |
    v
  拼接所有头 → 线性投影 → 输出
```

**关键点：多头不增加计算量！** 如果 d_model=512 且有 8 个头，每个头只处理 512/8=64 维。总计算量和单头 512 维一样。

> 下一章（第6章 Transformer 架构全景）将完整实现 Multi-Head Attention。

In [ ]:
# =============================================
# 8.1 简单演示多头注意力的效果
# =============================================
# 模拟 4 个注意力头，每个头有不同的 Q、K 权重
# 通过人为加入不同的偏置，模拟不同头学到不同的关注模式

n_heads = 4
sentence = ["The", "black", "cat", "sat", "on", "the", "mat"]
n = len(sentence)

fig, axes = plt.subplots(1, n_heads, figsize=(20, 4.5))

head_names = ['Head 1: Syntax', 'Head 2: Modifier', 'Head 3: Position', 'Head 4: Global']

torch.manual_seed(0)
for h, (ax, name) in enumerate(zip(axes, head_names)):
    # 每个头随机生成不同的 Q, K 向量（模拟不同的线性变换）
    Q_h = torch.randn(n, 8)
    K_h = torch.randn(n, 8)

    # 人为加入结构化偏差，模拟不同头学到不同的语言模式
    if h == 0:  # 头 1: 语法关系 — 主语关注谓语
        Q_h[0] += K_h[3]   # "The" 关注 "sat"
        Q_h[2] += K_h[3]   # "cat" 关注 "sat"
    elif h == 1:  # 头 2: 修饰关系 — 名词关注形容词
        Q_h[2] += K_h[1] * 3  # "cat" 强烈关注 "black"
    elif h == 2:  # 头 3: 位置关系 — 关注前一个词
        for i in range(n):
            if i > 0: Q_h[i] += K_h[i-1] * 2
    else:  # 头 4: 全局关系 — 关注句首
        for i in range(n):
            Q_h[i] += K_h[0] * 1.5

    # 计算Attention Weight
    scores = Q_h @ K_h.T / math.sqrt(8)  # 缩放点积
    weights = F.softmax(scores, dim=-1).detach().numpy()

    # 画热力图
    im = ax.imshow(weights, cmap='Blues', aspect='auto', vmin=0, vmax=0.5)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(sentence, fontsize=9, rotation=45)
    ax.set_yticklabels(sentence, fontsize=9)
    ax.set_title(name, fontsize=11)

plt.suptitle('Multi-Head Attention: Each Head Learns Different Patterns', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("每个注意力头关注不同的语言关系:")
print("  Head 1: 语法关系 (主语→谓语)")
print("  Head 2: 修饰关系 (名词→形容词)")
print("  Head 3: 位置关系 (关注前一个词)")
print("  Head 4: 全局关系 (关注句首)")
print("\n多头注意力 = 多个视角同时分析句子!")

---
## 9. 本章总结

**1. 为什么需要注意力** — 没有注意力，所有信息被压缩成一个向量（信息瓶颈）。有了注意力，每个词可以直接回头查看所有其他词

**2. Q、K、V 三要素** — Query（我在找什么）、Key（我有什么标签）、Value（我携带什么信息）。类比：搜索引擎查询 / 数据库 SELECT 语句

**3. Scaled Dot-Product Attention** — 公式：`softmax(QK^T / sqrt(d_k)) @ V`。三步：计算相关度 → 归一化成概率 → 加权提取信息

**4. Self-Attention** — Q、K、V 全部来自同一个序列。每个词都能直接看到所有其他词，解决了远距离依赖问题

**5. Masked Attention** — 因果遮罩（下三角矩阵）防止模型看到未来的词。GPT 用 Masked Self-Attention，BERT 用 Bidirectional Self-Attention

**6. 多头注意力** — 多个注意力头同时运行，每个头学习不同的关注模式（语法、指代、修饰等）。不增加总计算量

### 下一章预告

在第6章中，我们将学习 **Transformer 架构全景**：
- Transformer 的完整结构（Encoder + Decoder）
- 多头注意力的完整实现
- Layer Normalization 和残差连接
- 位置编码的作用

---

> **参考资料**
> - Vaswani et al., 2017: *Attention Is All You Need* — Transformer 原始论文
> - Jay Alammar: *The Illustrated Transformer* — Transformer 可视化讲解
> - Lilian Weng: *Attention? Attention!* — 注意力机制综述
> - 3Blue1Brown: *Attention in transformers, visually explained* — 可视化讲解视频